# Flow Matching FM-OT sur CIFAR-10 — notebook Colab complet (resit)

Pipeline **de bout en bout, en un seul notebook**, à exécuter sur **Google Colab avec GPU NVIDIA** :
setup → entraînement FM-OT (modèle réduit ~6,9 M) → FID → **ablation rectitude** (axes A et B) → figures.

1. **Setup** (GPU, dépôt, dépendances, Drive)
2. **Config**
3. **Entraînement FM-OT** (~150 k steps, reprise auto depuis Drive)
4. **FID(50 k) + NFE moyen (dopri5)**
5. **Métrique de rectitude $C$** (intrinsèque)
6. **Axe A** — $C$ et FID vs NFE / solveur
7. **Axe B** — $C$ au fil de l'entraînement (sur les checkpoints)
8. **Trajectoires FM-OT**
9. **Export des résultats**

> `Exécution → Modifier le type d'exécution → GPU` avant de lancer. Les checkpoints et figures sont
> écrits sur Google Drive (reprise automatique si la session Colab coupe).

## 0. Setup — GPU, dépôt, dépendances, Drive

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB)

# Le PACKAGE est chargé depuis GitHub (clone). Le Drive ne sert qu'au STOCKAGE des résultats.
REPO_URL    = "https://github.com/Alexis5131/Conditional-Flow-Matching.git"
REPO_BRANCH = "main"
LOCAL_REPO  = "/content/flow-matching-b3"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")  # uniquement pour persister checkpoints / figures / results

    if not os.path.exists(LOCAL_REPO):
        print(f"Clone du dépôt depuis GitHub ({REPO_BRANCH})...")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, LOCAL_REPO], check=True)
    else:
        subprocess.run(["git", "-C", LOCAL_REPO, "pull", "--ff-only"], check=False)
    subprocess.run(["pip", "install", "-q", "-e", LOCAL_REPO], check=True)
    sys.path.insert(0, f"{LOCAL_REPO}/src")
else:
    sys.path.insert(0, os.path.abspath("../src"))

import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0),
          "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("ATTENTION : aucun GPU détecté. Active un GPU NVIDIA "
          "(Exécution → Modifier le type d'exécution → GPU) avant l'entraînement.")

## 1. Config

In [ ]:
from pathlib import Path

RUN_NAME    = "fm-ot-cifar10"
MAX_STEPS   = 150_000
WARMUP      = 5_000
BATCH_SIZE  = 64
ACCUM_ITER  = 4          # batch effectif = 256
CKPT_EVERY  = 25_000     # checkpoints (servent à l'axe B)
EVAL_EVERY  = 50_000     # FID mi-entraînement (10 k samples)
USE_WANDB   = False

device = "cuda" if torch.cuda.is_available() else "cpu"

# Stockage des RÉSULTATS sur Drive (le code, lui, vient de GitHub).
RESULTS_ROOT = Path("/content/drive/MyDrive/flow-matching-b3") if IN_COLAB else Path("./runs")
RUN_DIR   = RESULTS_ROOT / RUN_NAME          # checkpoints + results.json
DATA_ROOT = Path("/content/data") if IN_COLAB else Path("./data")
PICS      = RUN_DIR / "pics"                 # figures (résultats) -> à copier dans report/pics pour le PDF
RUN_DIR.mkdir(parents=True, exist_ok=True); PICS.mkdir(parents=True, exist_ok=True)
print("run dir :", RUN_DIR)
print("figures :", PICS, "(copier dans report/pics/ du dépôt pour compiler le PDF)")

## 2. Helper FID

FID `clean-fid` en protocole `legacy_tensorflow` (seul comparable au papier). Le mid-training utilise
10 k échantillons (biaisé ↑ vs les 50 k de la table finale) ; la table finale en utilise 50 k.

In [ ]:
from flow_matching_b3.paths import get_path
from flow_matching_b3.sampling import sample
from flow_matching_b3.fid import compute_fid_cifar10
from flow_matching_b3.train import denorm

ot_path = get_path("ot", sigma_min=1e-4)

def make_fid_fn(path, n_samples=10_000, batch=128, solver="dopri5"):
    @torch.no_grad()
    def _fid(model):
        def gen():
            remaining = n_samples
            while remaining > 0:
                b = min(batch, remaining)
                imgs, _ = sample(model, path, shape=(b, 3, 32, 32), solver=solver,
                                 device=next(model.parameters()).device)
                yield denorm(imgs)
                remaining -= b
        return compute_fid_cifar10(gen(), n_samples=n_samples)
    return _fid

## 3. Entraînement FM-OT

Modèle réduit `adm_unet_cifar10_small` (~6,9 M params), chemin OT, ~150 k steps. La reprise depuis le
dernier checkpoint sur Drive est automatique : si Colab coupe, relancer cette cellule continue le run.

In [ ]:
import matplotlib.pyplot as plt
from flow_matching_b3.train import TrainConfig, train, find_latest_ckpt, make_grid_png
from flow_matching_b3.unet import adm_unet_cifar10_small
from flow_matching_b3.ema import EMA

cfg = TrainConfig(
    path_type="ot",
    run_dir=RUN_DIR,
    data_root=DATA_ROOT,
    max_steps=MAX_STEPS,
    batch_size=BATCH_SIZE,
    accum_iter=ACCUM_ITER,
    warmup_steps=WARMUP,
    poly_decay=True,
    ema_decay=0.9999,
    ckpt_every=CKPT_EVERY,
    log_every=200,
    eval_every=EVAL_EVERY,
    eval_n_samples=10_000,
    device=device,
    dtype="bfloat16" if device == "cuda" else "float32",
    use_wandb=USE_WANDB,
    wandb_run_name=RUN_NAME,
)

fid_fn = make_fid_fn(ot_path, n_samples=cfg.eval_n_samples)
final_ckpt = train(cfg, fid_fn=fid_fn)
print("Entraînement terminé. Checkpoint:", final_ckpt)

# Grille d'échantillons finaux (poids EMA, dopri5)
model = adm_unet_cifar10_small().to(device)
ema = EMA(model, decay=cfg.ema_decay)
state = torch.load(find_latest_ckpt(RUN_DIR), map_location=device)
model.load_state_dict(state["model"]); ema.load_state_dict(state["ema"])
with ema.swap_into(model):
    model.eval()
    grid_samples, _ = sample(model, ot_path, shape=(64, 3, 32, 32), solver="dopri5", device=device, seed=0)
grid = make_grid_png(grid_samples, n_row=8).permute(1, 2, 0).cpu()
plt.figure(figsize=(8, 8)); plt.imshow(grid); plt.axis("off"); plt.title(f"{RUN_NAME} — échantillons finaux")
plt.savefig(PICS / "final_samples_ot.png", bbox_inches="tight", dpi=120); plt.show()
del model, ema, state
if device == "cuda": torch.cuda.empty_cache()

## 4. Charger le modèle FM-OT (poids EMA) + FID(50 k) / NFE

In [ ]:
import numpy as np
from tqdm.auto import tqdm
from flow_matching_b3.metrics import straightness
from flow_matching_b3.sampling import make_vf, euler_sample, midpoint_sample, rk4_sample
from flow_matching_b3.train import enable_cuda_perf
enable_cuda_perf()

def load_ema_model(ckpt_path):
    model = adm_unet_cifar10_small().to(device)
    ema = EMA(model)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state["model"]); ema.load_state_dict(state["ema"])
    for p, p_ema in zip(model.parameters(), ema.shadow.parameters(), strict=True):
        p.data.copy_(p_ema.data)
    model.eval()
    return model, state["step"]

model_ot, step = load_ema_model(find_latest_ckpt(RUN_DIR))
print("FM-OT chargé @ step", step)

@torch.no_grad()
def eval_fid_and_nfe(model, path, n_samples=50_000, batch=128):
    nfes = []
    def gen():
        remaining = n_samples
        with tqdm(total=n_samples) as pbar:
            while remaining > 0:
                b = min(batch, remaining)
                imgs, nfe = sample(model, path, shape=(b, 3, 32, 32), solver="dopri5", device=device)
                nfes.append(nfe); pbar.update(b); yield denorm(imgs)
                remaining -= b
    return compute_fid_cifar10(gen(), n_samples=n_samples), float(np.mean(nfes))

fid_ot, nfe_ot = eval_fid_and_nfe(model_ot, ot_path)
print(f"FM-OT  FID(50k)={fid_ot:.3f}  NFE(dopri5)={nfe_ot:.1f}")

## 5. Métrique de rectitude $C$ (notre contribution)

Déviation moyenne de la trajectoire intégrée à la corde droite bruit→donnée, normalisée :

$$C = \frac{1}{N}\sum_i \frac{1}{K}\sum_k \big\lVert x^{(i)}_{t_k} - [(1-s_k)x^{(i)}_0 + s_k x^{(i)}_1]\big\rVert_2 \Big/ \lVert x^{(i)}_1 - x^{(i)}_0\rVert_2.$$

In [ ]:
C_intrinsic = straightness(model_ot, ot_path, shape=(3, 32, 32),
                           n_traj=512, nfe=200, solver="euler", device=device, seed=0)
print(f"C intrinsèque (FM-OT, Euler 200 pas, 512 traj) = {C_intrinsic:.4f}")

## 6. Axe A — rectitude et FID en fonction du NFE / solveur

Si le champ FM-OT est presque droit ($C$ petit), un intégrateur d'ordre faible à petit NFE doit déjà
donner un bon FID. On mesure conjointement $C$ et le FID (10 k samples, biaisé ↑ — lecture relative).

In [ ]:
NFE_GRID = [4, 8, 16, 32, 64]
SOLVERS = {"euler": euler_sample, "midpoint": midpoint_sample, "rk4": rk4_sample}
fid_curves = {s: [] for s in SOLVERS}
C_curves = {s: [] for s in SOLVERS}
vf = make_vf(model_ot, ot_path)

for sname, sfn in SOLVERS.items():
    for nfe in NFE_GRID:
        C = straightness(model_ot, ot_path, shape=(3, 32, 32),
                         n_traj=512, nfe=nfe, solver=sname, device=device, seed=0)
        def gen():
            remaining = 10_000
            while remaining > 0:
                b = min(256, remaining); x0 = torch.randn(b, 3, 32, 32, device=device)
                yield denorm(sfn(vf, x0, nfe=nfe)); remaining -= b
        f = compute_fid_cifar10(gen(), n_samples=10_000)
        fid_curves[sname].append(f); C_curves[sname].append(C)
        print(f"  {sname:8s} NFE={nfe:3d} -> C={C:.4f}  FID={f:.2f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
for s in SOLVERS:
    a1.plot(NFE_GRID, fid_curves[s], marker="o", label=s)
    a2.plot(NFE_GRID, C_curves[s], marker="o", label=s)
a1.set_xlabel("NFE"); a1.set_ylabel("FID (10k)"); a1.set_title("FM-OT — FID vs NFE"); a1.set_xscale("log"); a1.grid(alpha=0.3); a1.legend()
a2.set_xlabel("NFE"); a2.set_ylabel("rectitude C"); a2.set_title("FM-OT — C vs NFE"); a2.set_xscale("log"); a2.grid(alpha=0.3); a2.legend()
plt.tight_layout(); plt.savefig(PICS / "fig_rectitude_nfe.png", dpi=120, bbox_inches="tight"); plt.show()

## 7. Axe B — rectitude au fil de l'entraînement

$C$ calculé sur chaque checkpoint : le champ marginal appris se **redresse**-t-il vers la droite OT
idéale en convergeant ? (Pas de FID ici → peu coûteux.)

In [ ]:
ckpts = sorted(RUN_DIR.glob("ckpt_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
steps_B, C_B = [], []
for ck in ckpts:
    m, st = load_ema_model(ck)
    C = straightness(m, ot_path, shape=(3, 32, 32), n_traj=512, nfe=100, solver="euler", device=device, seed=0)
    steps_B.append(st); C_B.append(C)
    print(f"  step {st:>7} -> C={C:.4f}")
    del m
    if device == "cuda": torch.cuda.empty_cache()

if steps_B:
    plt.figure(figsize=(6, 4))
    plt.plot(steps_B, C_B, marker="o")
    plt.xlabel("step d'entraînement"); plt.ylabel("rectitude C"); plt.title("FM-OT — C au fil de l'entraînement")
    plt.grid(alpha=0.3); plt.savefig(PICS / "fig_rectitude_train.png", dpi=120, bbox_inches="tight"); plt.show()
else:
    print("aucun checkpoint ckpt_*.pt trouvé")

## 8. Trajectoires illustratives FM-OT

In [ ]:
@torch.no_grad()
def euler_trajectory(model, path, x0, t_stops=(0.0, 1/3, 2/3, 1.0), nfe=200):
    vf = make_vf(model, path); dt = 1.0 / nfe; x = x0.clone()
    out = {0.0: x.clone()}; next_idx = 1
    for k in range(nfe):
        t = torch.full((x.size(0),), k * dt, device=x.device)
        x = x + dt * vf(x, t)
        if next_idx < len(t_stops) and (k + 1) * dt >= t_stops[next_idx] - 1e-6:
            out[t_stops[next_idx]] = x.clone(); next_idx += 1
    return out

torch.manual_seed(7); x0 = torch.randn(8, 3, 32, 32, device=device)
stops = (0.0, 1/3, 2/3, 1.0)
traj = euler_trajectory(model_ot, ot_path, x0, t_stops=stops, nfe=200)
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for col, t in enumerate(stops):
    grid = make_grid_png(traj[t], n_row=8).permute(1, 2, 0).cpu()
    axes[col].imshow(grid); axes[col].axis("off"); axes[col].set_title(f"t = {t:.2f}")
plt.suptitle("FM-OT — trajectoire de sampling (Euler 200 pas)")
plt.savefig(PICS / "fig_traj_ot.png", dpi=120, bbox_inches="tight"); plt.show()

## 9. Export des résultats (à coller dans le rapport)

In [ ]:
import json
results = {"fid_50k": fid_ot, "nfe_dopri5": nfe_ot, "C_intrinsic": C_intrinsic,
           "axisA": {"nfe_grid": NFE_GRID, "fid": fid_curves, "C": C_curves},
           "axisB": {"steps": steps_B, "C": C_B}}
json.dump(results, open(RUN_DIR / "results.json", "w"), indent=2)

lines = ["| Modèle | FID (notre repro, 50k) | FID (papier, 38,3M) | NFE moyen (dopri5) | C (rectitude) |",
         "|---|---|---|---|---|",
         f"| FM-OT | {fid_ot:.2f} | 6.35 | {nfe_ot:.1f} | {C_intrinsic:.3f} |"]
print("\n".join(lines))
print("\n_Figures écrites dans report/pics/. Résultats complets dans", RUN_DIR / "results.json", "_")